# Import des différents modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import missingno as msno
import re
import seaborn as sns
import matplotlib.pyplot as plt

import bentoml

In [2]:
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler,  RobustScaler, MinMaxScaler

#Modèles
from sklearn.dummy import DummyRegressor

from sklearn.svm import SVR

from sklearn.ensemble import AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor, RandomForestRegressor

from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet

from sklearn.tree import DecisionTreeRegressor

# ajout
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline



### Lecture du fichier analysé

In [3]:
df_analyse_ml = pd.read_csv('./data/projet6_analyse.csv')


Lecture du fichier estimation

In [4]:
df_esti = pd.read_csv('./data/projet6_estimation.csv')
#df_esti = pd.read_json('./data/projet6_estimation.json' , orient = "records")

In [5]:
print (df_analyse_ml.isna().sum() )

DataYear                    0
BuildingType                0
PrimaryPropertyType         0
Latitude                    0
Longitude                   0
NumberofBuildings           0
PropertyGFATotal            0
PropertyGFAParking          0
LargestPropertyUseType      0
ENERGYSTARScore             0
TotalGHGEmissions           0
ENERGYSTARScoreIsMissing    0
Ratio_Electricity           0
Ratio_Steam                 0
BuildingAge                 0
mean_GFA_per_floor          0
Number_of_Use_Types         0
dtype: int64


## Comparaison des méthodes



**RMSE** (Root Mean Squared Error) : Mesure la taille moyenne des écarts entre valeurs prédites et réelles (dans l'unité de la variable).Objectif : À minimiser (plus il est proche de 0, plus le modèle est exact).

**MSE** (Mean Squared Error) : Moyenne des carrés des erreurs, correspondant à la variance résiduelle. Sert de base mathématique à minimiser lors de l'entraînement d'une régression. Penalise plus fortement les grands écarts.Objectif : À minimiser.

**MAE** (Mean Absolute Error) : Moyenne des valeurs absolues des écarts. Donnes une mesure directe de l'erreur moyenne sans sur-pénaliser les valeurs aberrantes.Objectif : À minimiser.

**R2** (Coefficient de détermination) : Mesure la qualité de la corrélation entre les prédictions et la réalité (exprime la proportion de variance expliquée).Objectif : À maximiser (plus il est proche de 1, plus les prédictions sont fidèles aux données réelles).


In [6]:
df_test1 = df_analyse_ml.copy()
# ==============================================================================
# 1. PRÉPARATION DES DONNÉES (X et y)
# ==============================================================================
# La cible y avec transformation log (lissage de l'asymétrie)
y = np.log1p(df_test1["TotalGHGEmissions"])
#y = df_test1["TotalGHGEmissions"]

# Les features X (tout sauf la cible)
X = df_test1.drop(columns=["TotalGHGEmissions"])

num_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
cat_features = X.select_dtypes(
    include=["category","str","object"]
).columns.tolist()


# ==============================================================================
# 2. PRÉPARATEUR DE DONNÉES (ColumnTransformer)
# ==============================================================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)


# ==============================================================================
# 3. DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)
# ==============================================================================
# 5 plis (folds), avec mélange aléatoire pour éviter tout biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# ==============================================================================
# 4. DICTIONNAIRE DES ALGORITHMES À TESTER
# ==============================================================================

models = {
    "Régression Linéaire": LinearRegression(),
    "Régression Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, random_state=42
    ),
}


# ==============================================================================
# 5. ÉVALUATION PAR VALIDATION CROISÉE POUR CHAQUE ALGORITHME
# ==============================================================================
results = []

for name, model in models.items():
    # Création du Pipeline complet (Préprocessing + Modèle)
    full_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )

    # découpage
    # Cross-validation évaluant le R² et la RMSE (Root Mean Squared Error)
    cv_results = cross_validate(
        full_pipeline,
        X,
        y,
        cv=kf,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        return_train_score=False,
    )

    # Récupération des moyennes des métriques
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    rmse_mean = -cv_results["test_rmse"].mean()
    mae_mean = -cv_results["test_mae"].mean()

    results.append(
        {
            "Algorithme": name,
            "R² (Moyenne)": round(r2_mean, 4),
            "R² (Écart-type)": round(r2_std, 4),
            "RMSE (Moyenne)": round(rmse_mean, 4),
            "MAE (Moyenne)": round(mae_mean, 4)
        }
    )

# Affichage du tableau récapitulatif
df_results = pd.DataFrame(results)
display(df_results)

,Algorithme,R² (Moyenne),R² (Écart-type),RMSE (Moyenne),MAE (Moyenne)
0,Régression Linéaire,0.6305,0.1341,0.7419,0.5351
1,Régression Ridge,0.6408,0.1312,0.7315,0.5278
2,Random Forest,0.7655,0.0264,0.6007,0.4538
3,Gradient Boosting,0.8113,0.0181,0.5393,0.4060


In [7]:
df_test2 = df_analyse_ml.copy()
# PRÉPARATION DES DONNÉES (X et y)
# La cible y avec transformation log (lissage de l'asymétrie)
y = np.log1p(df_test2["TotalGHGEmissions"])
#y = df_test2["TotalGHGEmissions"]
# Les features X (tout sauf la cible)
X = df_test2.drop(columns=["TotalGHGEmissions"])

num_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
cat_features = X.select_dtypes(
    include=["category","str","object"]
).columns.tolist()


# PRÉPARATEUR DE DONNÉES (ColumnTransformer)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)



#  DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)

# 5 plis (folds), avec mélange aléatoire pour éviter tout biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# DICTIONNAIRE DES ALGORITHMES À TESTER

models = {
    'dummy_reg': DummyRegressor(),
    'Régression Linéaire': LinearRegression(),
    'Régression Ridge' : Ridge(alpha=1.0),
    'lasso' : Lasso(random_state=42),
    'dec_tree':  DecisionTreeRegressor(random_state=42),
    'svr' : SVR(),
    'adaboost' :AdaBoostRegressor(random_state=42),
    'bagging': BaggingRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Random Forest' : RandomForestRegressor(n_estimators=100, random_state=42),
    'Knregressor' : KNeighborsRegressor()
}

#  ÉVALUATION PAR VALIDATION CROISÉE POUR CHAQUE ALGORITHME

results = []

for name, model in models.items():
    # Création du Pipeline complet (Préprocessing + Modèle)
    full_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )

    # découpage
    # Cross-validation évaluant le R² et la RMSE (Root Mean Squared Error)
    cv_results = cross_validate(
        full_pipeline,
        X,
        y,
        cv=kf,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        return_train_score=False,
    )

    # Récupération des moyennes des métriques
    r2_mean = cv_results["test_r2"].mean()
    r2_std = cv_results["test_r2"].std()
    rmse_mean = -cv_results["test_rmse"].mean()
    mae_mean = -cv_results["test_mae"].mean()

    results.append(
        {
            "Algorithme": name,
            "R² (Moyenne)": round(r2_mean, 4),
            "R² (Écart-type)": round(r2_std, 4),
            "RMSE (Moyenne)": round(rmse_mean, 4),
            "MAE (Moyenne)": round(mae_mean, 4)
        }
    )

# Affichage du tableau récapitulatif
df_results = pd.DataFrame(results)
display(df_results)

,Algorithme,R² (Moyenne),R² (Écart-type),RMSE (Moyenne),MAE (Moyenne)
0,dummy_reg,-0.0117,0.0094,1.2500,1.0132
1,Régression Linéaire,0.6305,0.1341,0.7419,0.5351
2,Régression Ridge,0.6408,0.1312,0.7315,0.5278
3,lasso,0.0361,0.0186,1.2199,0.9842
4,dec_tree,0.5591,0.0703,0.8228,0.6310
5,svr,0.0423,0.0079,1.2159,0.9827
6,adaboost,0.6706,0.0247,0.7125,0.5697
7,bagging,0.7443,0.0234,0.6277,0.4808
8,Gradient Boosting,0.8113,0.0181,0.5393,0.4060
9,Random Forest,0.7655,0.0264,0.6007,0.4538


In [8]:
df_test3 = df_analyse_ml.copy()

# PRÉPARATION DE X ET y

y = np.log1p(df_test3["TotalGHGEmissions"])

X = df_test3.drop(columns=["TotalGHGEmissions"])


#  DÉFINITION DE LA VALIDATION CROISÉE (K-Fold)
# 5 plis (folds), avec mélange aléatoire pour éviter tout biais d'ordre
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# SÉPARATION EN TRAIN (80%) ET TEST (20%) - JEU ISOLÉ

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Identification automatique des colonnes
num_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
cat_features = X_train.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

# Préparateur
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_features,
        ),
    ]
)

# Pipeline
full_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(random_state=42)),
    ]
)

# GRID SEARCH UNIQUEMENT SUR TRAIN (80%)
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5],
    "model__subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=kf,  # 5-Fold sur les 80%
    scoring="r2",
    n_jobs=-1,
    verbose=1,
)

print("--- Entraînement et optimisation sur le jeu Train (80%) ---")
grid_search.fit(X_train, y_train)

print(f"\n Meilleur score R² en CV (sur Train) : {grid_search.best_score_:.4f}")
print(f" Meilleurs hyperparamètres : {grid_search.best_params_}")


# On récupère le meilleur modèle entraîné
meilleur_model = grid_search.best_estimator_

# Prédictions sur les 20% jamais vus
y_pred_test = meilleur_model.predict(X_test)

# Calcul des métriques finales
r2_final = r2_score(y_test, y_pred_test)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_final = mean_absolute_error(y_test, y_pred_test)

print("\n" + "=" * 60)
print("RÉSULTATS DE VALIDATION FINALE SUR LE JEU DE TEST (20%)")
print("=" * 60)
print(f"R² (Test final)   : {r2_final:.4f}")
print(f"RMSE (Test final) : {rmse_final:.4f}")
print(f"MAE (Test final)  : {mae_final:.4f}")

--- Entraînement et optimisation sur le jeu Train (80%) ---
Fitting 5 folds for each of 54 candidates, totalling 270 fits

 Meilleur score R² en CV (sur Train) : 0.8241
 Meilleurs hyperparamètres : {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}

RÉSULTATS DE VALIDATION FINALE SUR LE JEU DE TEST (20%)
R² (Test final)   : 0.8062
RMSE (Test final) : 0.5792
MAE (Test final)  : 0.4252


### Résultat sur le data frame estimation 

In [9]:
df_test3 = df_esti .copy()

# PRÉPARATION DE DF_ESTI

# La valeur estimée réelle (passée en log1p comme pendant l'entraînement)
y_estime = np.log1p(df_test3["TotalGHGEmissions"])

# Les variables explicatives de df_esti
X_esti = df_test3.drop(columns=["TotalGHGEmissions"])


#  PRÉDICTION  MEILLEUR MODÈLE

y_pred_modele = meilleur_model.predict(X_esti)


# CALCUL DES MÉTRIQUES D'ALIGNEMENT

r2_accord = r2_score(y_estime, y_pred_modele)
mae_accord = mean_absolute_error(y_estime, y_pred_modele)

# Conversion de l'erreur brute (sur l'échelle réelle du CO2)
erreur_moyenne_co2 = np.expm1(mae_accord)

print("=" * 60)
print("ÉVALUATION DE LA QUALITÉ DES ESTIMATIONS DU DF_ESTI")
print("=" * 60)
print(f"R² (Modèle vs Estimation) : {r2_accord:.4f}")
print(
    f"Erreur Moyenne Absolue (MAE log)  : {mae_accord:.4f} (~{erreur_moyenne_co2:.2f} tCO2eq)"
)

ÉVALUATION DE LA QUALITÉ DES ESTIMATIONS DU DF_ESTI
R² (Modèle vs Estimation) : 0.9285
Erreur Moyenne Absolue (MAE log)  : 0.2218 (~0.25 tCO2eq)


### Sauvegarde du model avec bentoml

In [10]:
save_bentoml = bentoml.sklearn.save_model(name="seattle_co2" , model = meilleur_model )